In [ ]:
from read_pdf_experiment import *

documents_extractor1 = DocumentsExtractor("web_scraped_resources/matched_files", False)

c:\Users\louis\OneDrive\Documents\4th Year\Darwin\Hallucination Experiments Code\.venv\Lib\site-packages\langchain_community\document_loaders\parsers\pdf.py:154: UserWarning: Unknown PDF Filter!
  warnings.warn("Unknown PDF Filter!")


In [ ]:
__template__ = """Answer the question based on the context below. If you can't 
answer the question, reply "I don't know".

Context: {context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(__template__)
doc_extractor = documents_extractor1
embeddings = OpenAIEmbeddings()
parser = StrOutputParser()
vector_store  = DocArrayInMemorySearch.from_documents(
            doc_extractor.docs,
            embedding=embeddings,
        )
setup = RunnableParallel(context=vector_store.as_retriever(), question=RunnablePassthrough())

In [ ]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

from transformers import BertTokenizer, BertForMaskedLM, BertModel
from bert_score import BERTScorer

from evaluate import load

bertscore = load("bertscore")

prep_keys()

models = [
    'gpt-4o-2024-08-06', 'chatgpt-4o-latest', 'gpt-3.5-turbo-0125',
    'anthropic.claude-3-sonnet-20240229-v1:0', 'anthropic.claude-3-haiku-20240307-v1:0',
    'meta.llama3-70b-instruct-v1:0', 'meta.llama3-8b-instruct-v1:0',
    'mistral.mistral-large-2402-v1:0', 'mistral.mistral-7b-instruct-v0:2',
    'mistral.mixtral-8x7b-instruct-v0:1', 'gemini-2.0-flash-lite-preview-02-05',
    'gemini-1.5-flash', 'gemini-1.5-flash-8b', 'gemini-1.5-pro',
    'grok-2-vision-1212', 'grok-2-1212']

user_prompts = ["What is Project Gigabit?", "Summarise the Superfast Broadband Programme", "Summarise the role of BDUK"]

results = dict()
for user_prompt in user_prompts:
    results[user_prompt] = dict()
    for model_choice in models:
        if model_choice in ['gpt-4o-2024-08-06', 'chatgpt-4o-latest', 'gpt-3.5-turbo-0125']:
            model = ChatOpenAI(openai_api_key=os.getenv("OPENAI_API_KEY"), model = model_choice)
        elif model_choice in ['gemini-2.0-flash-lite-preview-02-05', 'gemini-1.5-flash', 'gemini-1.5-flash-8b', 'gemini-1.5-pro']:
            model = ChatGoogleGenerativeAI(model=model_choice, google_api_key=os.getenv('GEMINI_API_KEY'))
        elif model_choice in ['grok-2-vision-1212', 'grok-2-1212']:
            model = ChatXAI(xai_api_key=os.getenv("XAI_API_KEY"), model = model_choice)
        elif model_choice in ['anthropic.claude-3-sonnet-20240229-v1:0', 'anthropic.claude-3-haiku-20240307-v1:0',
                              'meta.llama3-70b-instruct-v1:0', 'meta.llama3-8b-instruct-v1:0',
                              'mistral.mistral-large-2402-v1:0', 'mistral.mistral-7b-instruct-v0:2',
                              'mistral.mixtral-8x7b-instruct-v0:1']:
            model = ChatBedrock(model_id=model_choice, aws_access_key_id=os.getenv('AWS_ACCESS_KEY_ID'), aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'))
        prompt = ChatPromptTemplate.from_template(__template__)
        chain = setup | prompt | model | parser
        #llmTester.set_model(model)
        response = chain.invoke(user_prompt)

        # BERT Score
        doc, _ = vector_store.similarity_search_with_score(query=user_prompt, k=10)[0]
        reference = doc.page_content
        candidate = response
        bert_results = bertscore.compute(predictions=[candidate], references=[reference], lang="en")

        # Rouge Score

        rouge_results = scorer.score(reference, candidate)

        results[user_prompt][model_choice] = {'Response': response, 'BERTScore': bert_results, 'Rouge Score': rouge_results}
results

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


{'What is Project Gigabit?': {'gpt-4o-2024-08-06': {'Response': "Project Gigabit is the UK government's national mission to deliver lightning-fast, reliable broadband for everyone in the country. It aims to upgrade the country's digital infrastructure to gigabit speeds, significantly enhancing connectivity for homes and businesses. The project involves a £5 billion investment to connect hard-to-reach and rural buildings with gigabit-capable broadband, while also promoting competition among telecom providers and working closely with various stakeholders such as industry, local councils, and consumer groups.",
   'BERTScore': {'precision': [0.9010617733001709],
    'recall': [0.7985973358154297],
    'f1': [0.8467410206794739],
    'hashcode': 'roberta-large_L17_no-idf_version=0.3.12(hug_trans=4.46.3)'},
   'Rouge Score': {'rougeL': Score(precision=0.569620253164557, recall=0.10975609756097561, fmeasure=0.18404907975460122)}},
  'chatgpt-4o-latest': {'Response': 'Project Gigabit is a UK 

In [11]:
import csv

# Load your data
data = results

# Prepare data for CSV
csv_data = []
headers = ['Question', 'Model', 'BERT Precision', 'BERT Recall', 'BERT F1', 'Rouge Precision', 'Rouge Recall', 'Rouge F1']

for question, models in data.items():
    for model, details in models.items():
        #response = details.get('Response', '')
        bert = details.get('BERTScore', {})
        rouge = details.get('Rouge Score', {}).get('rougeL')

        # Handle Score object or dictionary case
        if isinstance(rouge, dict):  # If it's a dictionary
            rouge_precision = rouge.get('precision', None)
            rouge_recall = rouge.get('recall', None)
            rouge_fmeasure = rouge.get('fmeasure', None)
        else:  # If it's an object
            rouge_precision = getattr(rouge, 'precision', None)
            rouge_recall = getattr(rouge, 'recall', None)
            rouge_fmeasure = getattr(rouge, 'fmeasure', None)

        csv_data.append([
            question, model,
            bert.get('precision', [None])[0], bert.get('recall', [None])[0], bert.get('f1', [None])[0],
            rouge_precision, rouge_recall, rouge_fmeasure
        ])

# Write to CSV file
csv_filename = 'results.csv'
with open(csv_filename, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(csv_data)

print(f"CSV file '{csv_filename}' successfully created!")




CSV file 'results.csv' successfully created!
